# Longitudinal bridges: I2SB vs ImMAP-SB

Two bridges to the same target (this session's CT1), run on the **same validation slices** with the
**same bridge noise**, so the arms are directly comparable:

1. **I2SB, T1 -> CT1** -- the ordinary bridge. The regressor
   (`I2SB_Unet_NYUMets_CT1_from_all_plus_otherstudy`) also *sees* the other study's CT1, but only as
   a conditioning channel; the bridge itself starts at this session's T1.
2. **ImMAP-SB, other CT1 -> CT1** -- the bridge starts at another study's CT1 (the wrong anatomy
   wherever the patient changed), and every endpoint estimate $\hat x$ is replaced by the learned
   data-consistency prox before the posterior update:

$$\tilde x = \arg\min_x \tfrac12\|x-\hat x\|^2/\gamma_t
            + \tfrac12\|M(\mathrm{T1} - A(x))\|^2/\sigma_A^2,
  \qquad \gamma_t = c\,\sigma_{\mathrm{eff}}(t)^2,$$

   solved by Gauss-Newton at $\hat x$ plus CG. $A$ is the frozen learned UNet CT1 -> T1. Nothing is
   trained here; this is the plug-and-play comparison.

Two reference arms are on by default, because arms 1 and 2 differ in *three* ways at once (net,
bridge start, prox) and a difference between them cannot otherwise be attributed:

3. **I2SB, other CT1 -> CT1** -- same net and same start as arm 2, no prox. Arm 2 minus arm 3 **is**
   the prox.
4. **copy x1** -- return the other study's CT1 unchanged. A prior scan already looks much like the
   current one away from the tumour, so this is the score a longitudinal method has to beat. The
   `rmse_chg` column -- the error where the two studies actually differ -- is where it should fail.

Everything is driven by the `ARMS` list: add, drop or reorder entries and the rest follows.
`refresh()` redraws a new random set of validation slices and re-runs every arm.

## Knobs

In [ ]:
import os, sys, math, time
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# ---- trained regressors: key -> that run's SAVED config -----------------------------------------
NETS = {
    # T1 -> CT1 bridge; C=5, conditioning = FLAIR, T1, T2, other-study CT1
    "all+other":  "trained_nets/nyumets/I2SB_Unet_NYUMets_CT1_from_all_plus_otherstudy/config.json",
    # other CT1 -> CT1 bridge; C=4, conditioning = FLAIR, T1, T2
    "from_other": "trained_nets/nyumets/I2SB_Unet_NYUMets_CT1_from_otherCT1/config.json",
}

# ---- what to run. start: "t1" | "other" (the bridge start). net=None -> return the start itself --
ARMS = [
    dict(name="I2SB  T1->CT1",       net="all+other",  start="t1",    prox=False),
    dict(name="ImMAP-SB  oCT1->CT1", net="from_other", start="other", prox=True),
    dict(name="I2SB  oCT1->CT1",     net="from_other", start="other", prox=False),
    dict(name="copy x1 (oCT1)",      net=None,         start="other", prox=False),
]
REF_ARM = 0                # per-slice differences are taken against this arm

# ---- the frozen forward operator A: CT1 -> T1 ---------------------------------------------------
A_CKPT = "trained_nets/nyumets/ForwardOp_UNet_w16_l3_xc_CT1T2FLAIR_to_T1/net.ckpt"
A_COND_IDX = [3, 0]        # A's side information (T2, FLAIR); ignored for a CT1-only A
A_DATA = {"image_key": "img_median_mad", "scales": [3.0, 3.0, 3.0, 3.0]}   # what A was trained on
SIGMA_A = None             # None = measure A's residual on a fixed calibration subset (below)

# ---- ImMAP-SB prox ------------------------------------------------------------------------------
C = 1.0                    # gamma_t = C * sigma_eff(t)^2   (0 reproduces I2SB exactly)
T_MAX = 1.0                # prox only for t <= T_MAX
CG_ITERS, CG_TOL, GN_ITERS = 10, 1e-4, 1
USE_MASK = True            # fidelity region M = brain mask (False: whole frame)

# ---- data and sampling --------------------------------------------------------------------------
NFE = None                 # None = each run's val_nfe
N_SLICES = 32              # val slices per refresh
BATCH = 8
SEED = 0                   # refresh() bumps this
SLICE_RANGE = (40, 110)    # original slice indices [lo, hi); None = the run's own setting
ENH_Q = 0.98               # enhancement proxy: top 2% of CT1 - T1 per slice
CHG_Q = 0.98               # interval-change proxy: top 2% of |CT1 - other CT1| per slice
CAL_SLICES, CAL_SEED = 64, 12345          # sigma_A calibration subset (independent of refresh)
N_SHOW = 4                 # slices drawn in the figures
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)

## Load the regressors and A

Each arm's net, schedule and sampler settings come from that run's own saved config. A missing
checkpoint drops its arms with a message rather than failing the notebook, so arm 1 is runnable
before the other-CT1 net has finished training.

All loaded runs must share one bridge schedule -- otherwise "the same noise draw" means nothing and
the arms are not paired.

In [ ]:
import datasets                                    # noqa: F401  (registers loaders)
from datasets.registry import build_loader
from models import build_model
from training.common import load_ckpt
from training.i2sb import _split_batch
from training.forward_op import fixed_val_subset, enh_region
from training.metrics import compute_metrics
from sb.base import build_schedule
from sb.i2sb import i2sb_sample
from sb.immap_sb import ImMAPProx, immap_sb
from sb.learned_dc import _load_E


def load_run(path):
    """A trained i2sb run -> everything needed to sample from it."""
    with open(path) as f:
        cfg = yaml.safe_load(f)
    if cfg.get("task") != "i2sb":
        raise ValueError(path + ": task is " + repr(cfg.get("task")) + ", not i2sb")
    ckpt = cfg["paths"].get("ckpt") or os.path.join(cfg["paths"]["save_dir"], "net.ckpt")
    if not os.path.exists(ckpt):
        raise FileNotFoundError("no checkpoint at " + ckpt)
    net = build_model(cfg).to(DEVICE).eval()
    load_ckpt(ckpt, model=net, device=DEVICE)
    i2 = cfg["i2sb"]
    sched = build_schedule(kind=i2.get("kind", "brownian"), tau=i2.get("tau", 0.19),
                           n_points=i2.get("n_points", 1000), beta_max=i2.get("beta_max", 0.3),
                           device=DEVICE)
    if hasattr(net, "assert_schedule_matches"):
        net.assert_schedule_matches(sched)
    v = cfg["data"]["val"]
    return dict(cfg=cfg, net=net, sched=sched,
                cond_idx=list(v.get("cond_idx") or []),
                guide_as_cond=bool(v.get("guide_as_cond")),
                data_range=float(cfg["training"].get("data_range", 1.0)),
                samp=dict(nfe=NFE or int(i2.get("val_nfe", 20)),
                          deterministic=bool(i2.get("deterministic", False)),
                          posterior=i2.get("posterior", "ddpm"),
                          clip_denoise=bool(i2.get("clip_denoise", False)), verbose=False))


RUNS = {}
for key, path in NETS.items():
    try:
        RUNS[key] = load_run(path)
        r = RUNS[key]
        print("loaded " + key + ": " + type(r["net"]).__name__
              + "  cond_idx=" + str(r["cond_idx"])
              + ("  +otherCT1" if r["guide_as_cond"] else "")
              + "  nfe=" + str(r["samp"]["nfe"]))
    except (FileNotFoundError, OSError) as e:
        print("SKIP " + key + ": " + str(e))

if not RUNS:
    raise RuntimeError("none of NETS could be loaded; nothing to sample from")
arms = [a for a in ARMS if a["net"] is None or a["net"] in RUNS]
dropped = [a["name"] for a in ARMS if a not in arms]
if dropped:
    print("dropped arms (net unavailable): " + ", ".join(dropped))

# one schedule, one nfe and one data_range, or the arms are not comparable
first = next(iter(RUNS))
ref = RUNS[first]
for k, r in RUNS.items():
    if not torch.allclose(r["sched"].std_fwd, ref["sched"].std_fwd):
        raise ValueError(k + " has a different bridge schedule from " + first)
    if r["samp"]["nfe"] != ref["samp"]["nfe"]:
        raise ValueError(k + ": val_nfe " + str(r["samp"]["nfe"]) + " != "
                         + str(ref["samp"]["nfe"]) + "; the arms would visit different steps and "
                         + "the paired noise would not line up. Set NFE to pin both.")
    if r["data_range"] != ref["data_range"]:
        raise ValueError(k + ": data_range " + str(r["data_range"]) + " != "
                         + str(ref["data_range"]) + "; the metrics would not be comparable")
sched0, data_range = ref["sched"], ref["data_range"]

A, a_rmse = _load_E(A_CKPT, DEVICE)
print("A: cond_channels=" + str(A.cond_channels)
      + ("  stored val rmse=%.4f" % a_rmse if a_rmse is not None else "  (no stored val rmse)"))

## Data

**One** loader feeds every arm, so all of them see the identical slices, the identical other-study
CT1 and the identical mask:

| batch key | image |
|---|---|
| `x0`   | this session's CT1 -- the target |
| `x1`   | another study's CT1 at the same original slice index (`guide_slice="index"`; the studies are registered) |
| `y`    | this session's T1 -- the bridge start for arm 1, and the measurement the prox enforces |
| `cond` | the union of every run's conditioning contrasts and A's |

Arm 1's net was trained with the other study's CT1 appended to `cond`; here it is handed `x1`, the
same image arm 2 bridges from, so the two arms cannot differ through *which* prior scan they saw.
`x1_source="other_study"` also drops patients with a single study in this split, so every arm is
scored on the same multi-study population.

In [ ]:
base = dict(ref["cfg"]["data"]["val"])
for k, v in A_DATA.items():
    if base.get(k) != v:
        raise ValueError("A was trained with " + k + "=" + str(v)
                         + " but the run's val data has " + str(base.get(k)))

ALL_COND = []
for r in RUNS.values():
    for c in r["cond_idx"]:
        if c not in ALL_COND:
            ALL_COND.append(c)
for c in (A_COND_IDX if A.cond_channels else []):
    if c not in ALL_COND:
        ALL_COND.append(c)
a_sel = [ALL_COND.index(c) for c in A_COND_IDX] if A.cond_channels else []

vcfg = dict(base)
vcfg.update(name="nyumets_guided", cond_idx=ALL_COND, batch_size=BATCH,
            # guides are assembled here, not by the loader: x1 IS the other-study CT1, so arm 1's
            # extra conditioning channel and arm 2's bridge start are the same image.
            guide_mode="none", guide_as_cond=False, guide_idx=None,
            x1_source="other_study", x1_other_idx=base.get("x0_idx", 2),
            y_idx=base.get("x1_idx", 1), guide_slice="index",
            deterministic=True, random_flips=False, crop_size=None, center_crop=None)
if SLICE_RANGE is not None:
    vcfg["slice_range"] = list(SLICE_RANGE)
full = build_loader(vcfg, shuffle=False, drop_last=False)


def cond_for(run, cond, other):
    """The conditioning stack THIS run was trained on, carved out of the shared one."""
    sel = [ALL_COND.index(c) for c in run["cond_idx"]]
    parts = ([cond[:, sel]] if sel else []) + ([other] if run["guide_as_cond"] else [])
    return torch.cat(parts, dim=1) if parts else None


print(str(len(full.dataset)) + " val slices | shared cond " + str(ALL_COND)
      + " | A cond " + str(A_COND_IDX if A.cond_channels else []))
for key, r in RUNS.items():
    want = getattr(r["net"], "C", None)
    got = 1 + len(r["cond_idx"]) + int(r["guide_as_cond"])
    flag = "" if (want is None or want == got) else "   <-- MISMATCH: net wants C=" + str(want)
    print("  " + key + ": net input channels = " + str(got) + flag)

### $\sigma_A$

$\sigma_A$ sets the prox's damping $\lambda_t = \sigma_A^2/\gamma_t$, so it has to be A's *actual*
residual on this data rather than a guess. The cell below measures
$\mathrm{RMS}\,[M(\mathrm{T1} - A(\mathrm{CT1}))]$ on a fixed calibration subset that `refresh()`
never touches, so the prox stays identical across refreshes.

It is measured on the validation split A was itself validated on, which makes it mildly optimistic;
for a plug-and-play diagnostic that is fine, and `SIGMA_A` overrides it.

In [ ]:
cal = fixed_val_subset(full, CAL_SLICES, CAL_SEED)
num = den = 0.0
with torch.no_grad():
    for batch in cal:
        x0, _, cond, mask, _, _ = _split_batch(batch, DEVICE)
        y = batch["y"].to(DEVICE)
        m = (mask > 0.5).float()
        cA = cond[:, a_sel] if a_sel else None
        num += float((((y - A(x0, cA)) ** 2) * m).sum())
        den += float(m.sum())
sigma_A_meas = math.sqrt(num / max(den, 1.0))
sigma_A = float(SIGMA_A) if SIGMA_A is not None else sigma_A_meas
print("sigma_A measured on %d calibration slices: %.4f" % (len(cal.dataset), sigma_A_meas)
      + ("   (stored in ckpt: %.4f)" % a_rmse if a_rmse is not None else "")
      + "   USING %.4f" % sigma_A)

prox = ImMAPProx(sched0, A, sigma_A, c=C, t_max=T_MAX, cg_iters=CG_ITERS, gn_iters=GN_ITERS,
                 cg_tol=CG_TOL)
print("prox: c=" + str(C) + "  t_max=" + str(T_MAX) + "  cg=" + str(CG_ITERS)
      + "  gn=" + str(GN_ITERS) + "  mask=" + str(USE_MASK))

## Sampling

`run_arms(seed)` draws a random subset of the validation set and runs every arm on it. The torch
RNG is reset to the same value before each arm, so all arms draw **identical** bridge noise --
whatever separates them is the method, not the draw.

In [ ]:
def change_region(x0, x1, m, q):
    """Per slice: brain pixels where |CT1 - other CT1| is in the top (1 - q) fraction.

    The sibling of training.forward_op.enh_region, taken on the INTERVAL difference instead of the
    contrast difference: this is where the two studies actually disagree, and the only region in
    which a longitudinal method can beat copying its own bridge start.
    """
    d = (x0 - x1).abs()
    out = torch.zeros_like(m)
    for i in range(x0.shape[0]):
        inb = m[i] > 0.5
        if inb.sum() < 10:
            continue
        thr = torch.quantile(d[i][inb].float(), q)
        out[i] = (inb & (d[i] >= thr)).float()
    return out


def _sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def run_arms(seed, n_slices=None, verbose=True):
    """-> (S, step_stats, seconds): the images, the prox diagnostics, the wall clock per arm."""
    loader = fixed_val_subset(full, n_slices or N_SLICES, seed)
    keep = {k: [] for k in ("x0", "x1", "y", "m", "enh", "chg", "cond_A")}
    preds = {a["name"]: [] for a in arms}
    step_stats = {a["name"]: [] for a in arms if a["prox"]}
    seconds = {a["name"]: 0.0 for a in arms}

    with torch.no_grad():
        for bi, batch in enumerate(loader):
            x0, x1, cond, mask, _, _ = _split_batch(batch, DEVICE)
            y = batch["y"].to(DEVICE)
            m = (mask > 0.5).float()
            cA = cond[:, a_sel] if a_sel else None
            bseed = seed * 100003 + bi

            for a in arms:
                start = y if a["start"] == "t1" else x1
                if a["net"] is None:                       # the copy baseline
                    preds[a["name"]].append(start.cpu())
                    continue
                run = RUNS[a["net"]]
                c_net = cond_for(run, cond, x1)
                torch.manual_seed(bseed)                   # paired noise across arms
                t0 = time.time()
                if a["prox"]:
                    out, _, _, st = immap_sb(run["net"], start, run["sched"], prox, y=y,
                                             cond=c_net, a_cond=cA,
                                             mask=m if USE_MASK else None, **run["samp"])
                    step_stats[a["name"]] += st
                else:
                    out, _, _ = i2sb_sample(run["net"], start, run["sched"], cond=c_net,
                                            **run["samp"])
                _sync()
                seconds[a["name"]] += time.time() - t0
                preds[a["name"]].append(out.real.cpu())

            keep["x0"].append(x0.cpu()); keep["x1"].append(x1.cpu()); keep["y"].append(y.cpu())
            keep["m"].append(m.cpu())
            keep["enh"].append(enh_region(x0, y, m, ENH_Q).cpu())
            keep["chg"].append(change_region(x0, x1, m, CHG_Q).cpu())
            keep["cond_A"].append(cA.cpu() if cA is not None else None)
            if verbose:
                print("batch " + str(bi + 1) + "/" + str(len(loader)), end="\r")

    S = {k: torch.cat(v) for k, v in keep.items() if k != "cond_A"}
    S["cond_A"] = None if not a_sel else torch.cat(keep["cond_A"])
    S["pred"] = {k: torch.cat(v) for k, v in preds.items()}
    return S, step_stats, seconds

## Metrics

Pooled over the brain mask.

* `rmse_enh` / `rmse_rest` -- the CT1 error inside and outside the enhancement proxy (top 2% of
  CT1 - T1). Enhancement lives in $A$'s null space, so the prox cannot create it; that stays the
  regressor's job, and these two columns say which of the two moved.
* `rmse_chg` -- the error where the two studies differ (top 2% of |CT1 - other CT1|). **This is the
  column that separates a real longitudinal model from a copy of the prior scan**: `copy x1` scores
  well everywhere else and has to fail here.
* `t1_res` -- consistency with the measurement, $\mathrm{RMS}\,[M(\mathrm{T1} - A(\cdot))]$.
  ImMAP-SB should pull it *toward* $\sigma_A$, not below it -- below means it is fitting A's own
  error.

In [ ]:
COLS = ["psnr", "ssim", "rmse", "rmse_enh", "rmse_rest", "rmse_chg", "t1_res", "sec"]


def summarize(S, name, seconds=None):
    """One arm's row of the table."""
    x0, y, m, enh, chg = S["x0"], S["y"], S["m"], S["enh"], S["chg"]
    s = S["pred"][name]
    rest = m * (1 - enh)
    e2 = (s - x0) ** 2
    rm = lambda w: math.sqrt(float((e2 * w).sum()) / max(float(w.sum()), 1.0))

    t1 = torch.zeros_like(x0)                    # chunked: A runs on the GPU, the stack does not
    with torch.no_grad():
        for i in range(0, x0.shape[0], BATCH):
            sl = slice(i, i + BATCH)
            c = None if S["cond_A"] is None else S["cond_A"][sl].to(DEVICE)
            t1[sl] = ((y[sl].to(DEVICE) - A(s[sl].to(DEVICE), c)) ** 2).cpu()

    mets = compute_metrics(x0 * m, s * m, data_range=data_range, mask=m)
    return {"psnr": float(mets["psnr"]), "ssim": float(mets["ssim"]), "rmse": rm(m),
            "rmse_enh": rm(enh), "rmse_rest": rm(rest), "rmse_chg": rm(chg),
            "t1_res": math.sqrt(float((t1 * m).sum()) / max(float(m.sum()), 1.0)),
            "sec": float(seconds.get(name, 0.0)) if seconds else 0.0}


def report(S, seconds=None):
    res = {a["name"]: summarize(S, a["name"], seconds) for a in arms}
    w = max(len(n) for n in res) + 1
    print(" " * w + " ".join("%9s" % c for c in COLS)
          + "     (sigma_A = %.4f, %d slices)" % (sigma_A, S["x0"].shape[0]))
    for name, r in res.items():
        print(("%-" + str(w) + "s") % name + " ".join("%9.4f" % r[c] for c in COLS))
    return res

## Figures

First figure: the inputs and every arm's sample, on one shared grayscale window taken from the
target. Second: each arm's error against CT1 on one shared diverging window, plus the
interval-change map -- errors that line up with the change map are the ones that matter.

In [ ]:
from visualization.image import subplot_images


def show_slices(S, n=None, idx=None):
    """Two rows-of-slices figures: the images, then the errors + the change map."""
    n = N_SHOW if n is None else n
    idx = list(range(min(n, S["x0"].shape[0]))) if idx is None else list(idx)
    names = [a["name"] for a in arms]

    rows, labels = [], []
    for i in idx:
        rows.append([S["y"][i, 0], S["x1"][i, 0], S["x0"][i, 0]]
                    + [S["pred"][k][i, 0] for k in names])
        labels.append("slice " + str(i))
    subplot_images(rows, row_labels=labels,
                   col_titles=["T1 (y)", "x1: other CT1", "CT1 (target)"] + names,
                   cmap="gray", window_from=[S["x0"][idx]], p=(1, 99),
                   mask=S["m"][idx], apply_mask=True, magnitude=False,
                   panel_size=(2.4, 2.6), show=False)
    plt.show()

    err = (S["pred"][names[REF_ARM]][idx] - S["x0"][idx])[S["m"][idx] > 0.5]
    v = float(torch.quantile(err.abs().float(), 0.99)) if err.numel() else 1.0
    rows, labels = [], []
    for i in idx:
        rows.append([S["pred"][k][i, 0] - S["x0"][i, 0] for k in names]
                    + [(S["x0"][i, 0] - S["x1"][i, 0]).abs()])
        labels.append("slice " + str(i))
    ncol = len(names)
    subplot_images(rows, row_labels=labels,
                   col_titles=[k + " - CT1" for k in names] + ["|CT1 - oCT1|"],
                   cmap=["RdBu_r"] * ncol + ["magma"],
                   vmin=[-v] * ncol + [0.0], vmax=[v] * ncol + [v],
                   share_window=False, mask=S["m"][idx], apply_mask=True,
                   magnitude=False, panel_size=(2.4, 2.6), show=False)
    plt.show()

## Refresh

**Re-run this cell for a new random set of validation slices.** `refresh()` bumps the seed each
time; `refresh(seed=...)` returns to a specific draw and `refresh(n_slices=...)` changes how many.
It leaves `S`, `step_stats`, `seconds` and `results` in the namespace, so the cells below can be
re-run against the new draw without re-sampling.

In [ ]:
_refresh_n = 0


def refresh(seed=None, n_slices=None, show=True):
    """New slices -> run every arm -> print the table and draw the figures."""
    global S, step_stats, seconds, results, cur_seed, _refresh_n
    if seed is None:
        _refresh_n += 1
        seed = SEED + _refresh_n
    cur_seed = seed
    t0 = time.time()
    S, step_stats, seconds = run_arms(seed, n_slices)
    print("seed " + str(seed) + " | " + str(S["x0"].shape[0]) + " slices | "
          + "%.1f s total" % (time.time() - t0) + " " * 20)
    results = report(S, seconds)
    if show:
        show_slices(S)
    return S


refresh(seed=SEED)

## Per slice

Is an arm's advantage systematic, or driven by a few slices? Per-slice PSNR of every other arm
minus the reference arm; mass to the right of zero means it wins on most slices.

In [ ]:
from visualization.hist import plot_hist


def per_slice_psnr(S, name):
    out = []
    for i in range(S["x0"].shape[0]):
        m = S["m"][i:i + 1]
        if float(m.sum()) == 0:
            out.append(np.nan)
            continue
        out.append(float(compute_metrics(S["x0"][i:i + 1] * m, S["pred"][name][i:i + 1] * m,
                                         psnr_only=True, data_range=data_range, mask=m)["psnr"]))
    return np.array(out)


names = [a["name"] for a in arms]
psnrs = {k: per_slice_psnr(S, k) for k in names}
base_name = names[REF_ARM]
series = {}
for k in names:
    if k == base_name:
        continue
    d = psnrs[k] - psnrs[base_name]
    d = d[np.isfinite(d)]
    series[k] = d
    print("%-22s vs %-22s: mean %+.3f dB, median %+.3f dB, better on %3.0f%% of %d slices"
          % (k, base_name, d.mean(), np.median(d), 100 * (d > 0).mean(), d.size))

plot_hist(series, bins=30, vlines={"no change": 0.0},
          xlabel="PSNR - PSNR(" + base_name + ")  [dB]",
          title="per-slice PSNR difference", show=True)

### How much did the patient actually change?

The copy baseline's error against CT1 *is* the interval change. If most slices sit near zero, the
two studies are nearly identical there and no method can show a gain -- the comparison only carries
signal in the right-hand tail. Read this before concluding that a flat metrics table means "no
difference between the methods"; it may mean "no difference between the studies".

In [ ]:
brain = S["m"] > 0.5
chg_rms = []
for i in range(S["x0"].shape[0]):
    b = brain[i]
    chg_rms.append(float((((S["x0"][i] - S["x1"][i]) ** 2)[b]).mean().sqrt())
                   if bool(b.any()) else np.nan)
chg_rms = np.array(chg_rms)
chg_rms = chg_rms[np.isfinite(chg_rms)]
print("per-slice RMS |CT1 - other CT1| over the brain: median %.4f, p90 %.4f, max %.4f"
      % (np.median(chg_rms), np.percentile(chg_rms, 90), chg_rms.max()))
plot_hist(chg_rms, bins=30, xlabel="RMS |CT1 - other CT1| over the brain",
          title="how different the two studies are, per slice", show=True)

## What the prox did along the bridge

Per visited step, for every arm that uses the prox: the damping $\lambda_t = \sigma_A^2/\gamma_t$,
the data residual $\|M(\mathrm{T1}-A(\cdot))\|$ before and after, and the RMS size of the
correction. Near $t=1$ -- the other study's CT1, where the start carries the wrong anatomy --
$\lambda_t$ is small and the prox is close to a pure fit of $A(x)=\mathrm{T1}$; near $t=0$ it is
large and the prox barely moves $\hat x$.

In [ ]:
n_pts = max(1, sched0.std_fwd.shape[0] - 1)
for name, st in step_stats.items():
    act = [s for s in st if s.get("active")]
    if not act:
        print(name + ": the prox was never active (C = 0, or T_MAX too small)")
        continue
    steps = sorted({s["step"] for s in act})
    agg = lambda key: [np.mean([s[key] for s in act if s["step"] == k]) for k in steps]
    t = [k / n_pts for k in steps]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    ax[0].semilogy(t, agg("lam"), "o-"); ax[0].set_title(r"damping $\lambda_t$")
    ax[1].plot(t, agg("res_before"), "o-", label="before prox (at $\\hat x$)")
    ax[1].plot(t, agg("res_after"), "s-", label="after prox")
    ax[1].axhline(sigma_A, ls="--", c="k", label=r"$\sigma_A$")
    ax[1].set_title(r"data residual $\|M(T1 - A)\|$"); ax[1].legend(fontsize=8)
    ax[2].plot(t, agg("delta_rms"), "o-"); ax[2].set_title(r"correction $\|\tilde x - \hat x\|$")
    for a in ax:
        a.set_xlabel("bridge position t  (0 = CT1, 1 = bridge start)"); a.grid(alpha=0.3)
    fig.suptitle(name)
    plt.tight_layout(); plt.show()

## Reading it

Read the table in this order:

1. **`copy x1` against everything else.** If a sampler cannot beat copying the prior scan on
   `rmse_chg`, the longitudinal bridge is a copy shortcut and nothing below matters. Check the
   change histogram first: if the studies barely differ on these slices, draw more with `refresh()`
   rather than concluding anything.
2. **Arm 2 against arm 3** (ImMAP-SB vs I2SB, same net, same start). This difference **is** the
   prox. Expect it in `rmse_rest` and `rmse_chg`, not in `rmse_enh` -- the prox anchors anatomy to
   T1, and enhancement is in $A$'s null space where the prox is blind.
3. **Arm 2 against arm 1** (the other-CT1 bridge vs the ordinary T1 bridge). This is the question
   the experiment exists to answer, but it moves the net, the start and the prox at once; only
   attribute it with 2 and 3 in hand.

Diagnostics:

* **`t1_res` well below $\sigma_A$**, or samples visibly drifting toward T1: the prox is too strong
  at large $t$. Lower `C`, or set `T_MAX` below 1.
* **Arm 2 identical to arm 3**: the prox never fired. Check `C > 0` and the per-step plots.
* **Arm 2 worse than arm 3 only in `rmse_enh`**: the prox is suppressing enhancement it cannot see.
  That is the failure mode to watch on this bridge, because enhancement inherited from the *other*
  study is equally invisible to $A$ and cannot be removed by data consistency either.
* For a sweep over `C` instead of a single value, `scripts/eval_immap_sb.py` runs several in one go.